This will be my rust learning log/diary! 

I initially placed some of these code blocks and notes into an obsidian .md file but will likely move them into here for convience!

In [ ]:
fn main(){
    println!("Hello, world!");
}

println!("Hello, world!"); // In jupyter this will print hello world without needing to be in a main function. 
                           // In a normal rust file this will cause an error because println! is not allowed outside of a function.

Hello, world!


The above code block visualises two scenarios. Rust is a compiled language, meaning all written code must be 'translated' in its entirety before it can be run. Where as in Python, an interpreted language, each line is 'translated' one line at a time so it can run each of those lines dynamically. 

This largely affects how errors in syntax + other errors are handled. Most errors will cause the Rust compilation to crash or fail, even if there is working code elsewhere. Whilst with Python the compilation per line won't necessarily fail in the same way. Certain lines of code that work will still run and work and only lines with errors will cause issues. 

---

Anatomy of a Rust Program: 

'''

fn main() {

}

'''

The main() function in Rust is unique in the fact that in any rust program it is always the first piece of code that runs. In the immediate example above, the () would contain parameters (if there were any). 


Regardless of the function, it is always wrapped in {}. This is required when wrapping around all function bodies. 
Good practice is to place the first { on the same line as the fn with a single space between () and {.  

Firstly, println! calls a rust macro. Had it called a function it would instead be entered as println without the !. Macros are a way to write code that generates code to extend Rust syntax. The use of ; demarks the end of the expression in the code.

---

### Cargo

Cargo is Rust's build system and package manager. Cargo handles a lot of tasks such as compiling code, downloading libraries/dependencies and building those libraries. 

---

### Programming a Guessing game!

In [2]:
use std::io; 

fn main() {
    println!("Guess the number!");

    println!("Please input your guess");

    let mut guess = String::new(); // this creates a mutable variable and assigns it to an empty string

    io::stdin()
        .read_line(&mut guess)
        .expect("Failed to read line");

    println!("You guessed: {guess}");
        
}

So the use of 'use std::io;', this imports and calls the input/output library. By default Rust has a set of items defined in the standard library that it brings into the scope of every program. Thi is called the 'Prelude'.

In Rust variables are immutable by default so the inclusion of mut is necessary to allow them to be changed. 

In [ ]:
let apples = 5; // immutable
let mut bananas = 5; // mutable

The :: syntax in the ::new line indicates that new is an associated function of the String type. An associated function is a function that is implemented on a type, in this case String.

In full, the let mut guess = String::new(); line has created a mutable variable that is currently bound to a new, empty instance of a String.

In [ ]:
    io::stdin()
        .read_line(&mut guess)

It is still possible to call io::stdin() without the use of the use std::io;. Instead we need to specify the module it comes from std::io::stdin

The .read_line(&mut guess) calls the read_line method on the standard input handle to get input from the user (In laymans terms, it provides a method for ths system to actually read the input). We are passing the arguments as to what string to store the input in (the &mut just tells the system to make a mutable reference to that input stored in the guess string). Must be mutable so we can change the strings content. 

The & indicates that the argument is a reference, which gives you a way to let multiple parts of the code access the data without needing to copy that data in memory multiple times. 

---

### Handling Potential Failure with Result:

Next element to examine is the '.expect("Failed to read line");'. This is still part of one logical line. 

As mentioned earlier, read_line puts whatever the user enters into the string we pass to it, but it also returns a *Result* value. The *Result* is an enumeration (enum for short), which is a type that can be stored in one of mutiple states (a process that has 3 states, queued, running and finished is a good example of an enum), each variant of the state in an enum can even hold data related to that specific state.

*Result*'s variants are Ok and Err. The Ok variant indicates the operation was successful, and it contains the successfully gneerated value. Err variants mean the operation has failed, and it contains the information regarding that failure. 

Values of the *Result*  type, like values of any type, have methods defined on them. For example an instance of *Result*  has an expect method that you can call. If this instance of *Result* is an Err value, expect will cuase the program to crash and display the error message you passed as the argument to expect. If the read_line method returns an Err, it would be likely be the result of an error coming from the underlying operating system. If this instance of Result is an Ok value, expect will take the return value that Ok is holding and return just that value to you so that you can use it. In this case, that value is the number of bytes in the user's input. 

---

##### Printing Values with println! Placeholders

In [ ]:
    println!("You guessed: {guess}");

The line prints the string that now contains the user's input. The {} set of is a placeholder. When printing the value of a variable, the variable name can go inside the curly brackets. When printing the result of evaluating an expression, place empty {} in the format string, then follow the format string with a comma-seperated list of expressions to print in each empty {} placeholder in the same order. Refer below for the example:

In [3]:
let x = 5;
let y = 10;

println!("x = {x} and y + 2 = {}", y + 2);

x = 5 and y + 2 = 12


--- 

#### Generating a secret number

In regards to the guessing game, we need to find a way to generate a random number that user will try to guess. The number needs to be different every time so that the game is unique each time. 

In [ ]:
    let secret_number = rand::thread_rng().gen_range(1..=100);

    println!("The secret number is: {secret_number}");

So, Rng trait defines methods that random number generators implement, and this trait must be in scope for us to use those methods. 

In the first new line added, we call rand::thread_rng function that gives us the particular random number generator we want to use: one that is local to the current thread of execution and is seeded by the operating system. Then, we call the gen_range method of the random number generator. This method is defined by the Rng trait that we brought into scope with the use rand::Rng;. The gen_range method takes a range expression as an argument and generates a random number in the range. The kind of range expression we’re using here takes the form start..=end and is inclusive on the lower and upper bounds, so we need to specify 1..=100 to request a number between 1 and 100.

#### Comparing the guess to the Secret number

Now that a mechanism to compare the guess to the randomly generated number.

In [ ]:
use std::cmp::Ordering; // brings the ordering in scope. 
use std::io;

use rand::Rng;

fn main() {
    // --snip--

    println!("You guessed: {guess}");

    match guess.cmp(&secret_number) {
        Ordering::Less => println!("Too small!"), // ordering is an enum with three variants: Less, Greater, and Equal
        Ordering::Greater => println!("Too big!"),
        Ordering::Equal => println!("You win!"),
    }
}

Depending on how the comparison is made, a different variant is returned. Then we can tell the programme to proceed based on the returned values from the comparison. The code above though will not compile due to mismatched types. Whe initially writing let mut guess = String::new(), rust intelligently guessed that guess should be a string. Whereas the secret_number is a number type. A few of Rust’s number types can have a value between 1 and 100: i32, a 32-bit number; u32, an unsigned 32-bit number; i64, a 64-bit number; as well as others. 

Ultimately we need to convert the string type to a number type. We can remedy this issue by casting a type to the guess. 
'let guess: u32 = guess.trim().parse().expect("Please type a number!");'

--- 

However, a strange element can be spotted. The reassigning of the variable 'guess'. Rust allows us to 'shadow' the previous value of guess with a new one. _Shadowing_ allows us to reuse the guess variable name rather than forcing us to create two unique variables (i.e guess_str and guess). We can bind this new variable to the expression 'guess.trim().parse()'. This reference to guess refers to the original value (pre-reassignment) this is the one that contained the orignal string. 

The trim method that is applied to the string will eliminate whitespace before and after the string being input. This is important because in order to convert the string to u32 we need to strip away the whitespace. The user must press 'enter'to satisfy read_line and input their guess, which adds a newline character to the the string. This is why we need to use the .trim method to remove that.

For example, if the user types 5 and presses enter, guess looks like this: 5\n. The \n represents “newline.” (On Windows, pressing enter results in a carriage return and a newline, \r\n.) The trim method eliminates \n or \r\n, resulting in just 5.

---

The parse method allows for the string (or whatever type) to be converted to another type. In the example here, we converted a string to a numeric type. We need to tell Rust what type to use by way of 'let guess: u32'. The colon(:) after guess tells rust we will annotate the variable's type. This then allows us to tell Rust we want the numeric type of of an unsigned, 32-bit integer. 

Additionally, the u32 annotation in this example program and the comparison with secret_number means Rust will infer that secret_number should be a u32 as well. So, now the comparison will be between two values of the same type!

---

#### Looping and allowing for multiple guesses

The 'loop'keyword creates an infinite loop. We can loop through the game. 

In [ ]:
    // --snip-- this just acts as a code snippet to allow for looping

println!("The secret number is: {secret_number}");

loop{
    println!("Please input your guess.");

    match guess.cmp(&secret_number) {
        Ordering::Less=>println!("too small!"),
        Ordering::Greater =>println!("too big!"),
        Ordering::Equal=>println!("You win!"),
    }
}